# 실습 과제 답안: 농산물 가격 정보 시각화 대시보드

같은 폴더에 **제공된 KAMIS Excel 파일**을 이용해 Streamlit 대시보드를 구현합니다.

| 항목 | 내용 |
|------|------|
| 목표 | 농산물 선택 → 평균 가격 / 가격 변화 그래프 / 원본 표 표시 |
| 데이터 | `딸기`, `배추`, `사과`, `수박`, `쌀` `_가격정보.xlsx` (제공 파일 사용) |
| 실행 | 노트북으로 데이터·로직 확인 후 `streamlit run app.py` |
| 선택 과제 | 최고·최저 가격, 전월 대비 변화율, 평균 가격 Bar Chart 비교 |

## 0. 라이브러리 설치 (필요 시)

아래 셀은 한 번만 실행하면 됩니다.

In [1]:
# 사용법: 과제에 필요한 패키지를 설치합니다.
%pip install pandas openpyxl plotly streamlit -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 제공 Excel 파일 구조 확인

KAMIS에서 내려받은 파일은 대략 다음 구조입니다.

| 행 | 내용 |
|----|------|
| 0행 | 제목 (품목/규격 설명) |
| 1행 | 컬럼명: 날짜, 가격, 등락률 |
| 2~4행 | 평년 월 / 전년 월 / 당월 평균 (요약) |
| 5행~ | `YYYY.MM.DD` 일자별 가격 |

대시보드에서는 **일자별 가격 행만** 사용합니다.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd()

PRODUCT_FILES = {
    "딸기": "딸기_가격정보.xlsx",
    "배추": "배추_가격정보.xlsx",
    "사과": "사과_가격정보.xlsx",
    "수박": "수박_가격정보.xlsx",
    "쌀": "쌀_가격정보.xlsx",
}

# 사용법: 제공 파일이 모두 있는지 먼저 확인합니다. (데이터 생성하지 않음)
missing = [f for f in PRODUCT_FILES.values() if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "다음 제공 Excel 파일이 없습니다. 같은 폴더에 파일을 두고 다시 실행하세요.\n- "
        + "\n- ".join(missing)
    )

raw = pd.read_excel(DATA_DIR / PRODUCT_FILES["딸기"], header=None)
print("원본 shape:", raw.shape)
print("제목:", raw.iloc[0, 0])
raw.head(8)

원본 shape: (61, 3)
제목: 소매가격 채소류, 딸기, 설향, 상품, 500g 정보 - 날짜, 가격, 등락률로 구성


,0,1,2
0,"소매가격 채소류, 딸기, 설향, 상품, 500g 정보 - 날짜, 가격, 등락률로 구성",NaN,NaN
1,날짜,가격,등락률
2,전월 순,0,∞
3,전년 순,0,∞
4,순 평년,0,∞
5,2026.05.21,"5,214",-2.52
6,2026.05.20,"5,349",-2.48
7,2026.05.19,"5,485",4.26


## 2. Excel 전처리 + 평균 가격 계산

- 요약 행(평년/전년/당월) 제외
- `5,214`처럼 쉼표가 있는 가격을 숫자로 변환
- 날짜를 `datetime`으로 변환

In [2]:
def to_number(series: pd.Series) -> pd.Series:
    """'5,214', '-' 같은 값을 숫자로 변환합니다."""
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"-": None, "nan": None, "None": None})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def load_price_data(product: str) -> pd.DataFrame:
    """제공 Excel에서 일자별 가격만 읽어 정리합니다."""
    file_path = DATA_DIR / PRODUCT_FILES[product]
    raw = pd.read_excel(file_path, header=None)

    df = raw.iloc[2:].copy()
    df.columns = ["날짜", "가격", "등락률"]

    date_mask = df["날짜"].astype(str).str.match(r"^\d{4}\.\d{2}\.\d{2}$", na=False)
    df = df.loc[date_mask].copy()

    df["날짜"] = pd.to_datetime(df["날짜"], format="%Y.%m.%d")
    df["가격"] = to_number(df["가격"])
    df["등락률"] = to_number(df["등락률"])
    df = df.dropna(subset=["가격"]).sort_values("날짜").reset_index(drop=True)
    return df


product = "딸기"
df = load_price_data(product)

avg_price = df["가격"].mean()
max_price = df["가격"].max()
min_price = df["가격"].min()

print(f"[{product}] 행 수: {len(df)}")
print(f"[{product}] 평균 가격: {avg_price:,.0f}원")
print(f"[{product}] 최고 가격: {max_price:,.0f}원")
print(f"[{product}] 최저 가격: {min_price:,.0f}원")

# 전월 대비 (월별 평균의 최근 두 달)
monthly = (
    df.assign(연월=df["날짜"].dt.to_period("M").astype(str))
    .groupby("연월", as_index=False)["가격"]
    .mean()
    .sort_values("연월")
)
prev = monthly.iloc[-2]["가격"]
curr = monthly.iloc[-1]["가격"]
change_rate = (curr - prev) / prev * 100
print(f"[{product}] 전월 대비: {change_rate:+.1f}%")
df.head()

[딸기] 행 수: 56
[딸기] 평균 가격: 5,731원
[딸기] 최고 가격: 7,147원
[딸기] 최저 가격: 5,061원
[딸기] 전월 대비: -1.7%


,날짜,가격,등락률
0,2026-03-03,7116,0.00
1,2026-03-04,7047,-0.97
2,2026-03-05,7147,1.42
3,2026-03-06,7046,-1.41
4,2026-03-09,7041,-0.07


## 3. Plotly로 가격 변화 Line Chart

- X축: `날짜`
- Y축: `가격`
- 제목: `{농산물} 가격 변화`

In [3]:
import plotly.express as px

fig = px.line(
    df,
    x="날짜",
    y="가격",
    markers=True,
    title=f"{product} 가격 변화",
)
fig.update_layout(xaxis_title="날짜", yaxis_title="가격(원)")
fig.show()

## 4. (선택) 5개 농산물 평균 가격 비교 — Bar Chart

In [4]:
rows = []
for name in PRODUCT_FILES:
    temp = load_price_data(name)
    rows.append({"농산물": name, "평균가격": temp["가격"].mean()})

avg_df = pd.DataFrame(rows)
display(avg_df)

fig_bar = px.bar(
    avg_df,
    x="농산물",
    y="평균가격",
    text_auto=".0f",
    title="5개 농산물 평균 가격 비교",
)
fig_bar.update_layout(yaxis_title="평균 가격(원)")
fig_bar.show()

,농산물,평균가격
0,딸기,5731.178571
1,배추,4008.855769
2,사과,27084.942308
3,수박,27088.557692
4,쌀,62305.413462


## 5. Streamlit 앱(`app.py`) 실행

같은 폴더의 `app.py`가 최종 대시보드입니다. 제공 Excel만 읽으며, 데이터를 새로 생성하지 않습니다.

```bash
streamlit run app.py
```

중지: 터미널에서 `Ctrl + C`

In [5]:
# 사용법: app.py가 제공 Excel을 읽는지 확인
from pathlib import Path

app_path = Path("app.py")
assert app_path.exists(), "app.py 가 없습니다."

code = app_path.read_text(encoding="utf-8")
assert "_make_sample_data" not in code
assert "to_excel" not in code
print("app.py OK — 제공 Excel 로딩 방식")
print("--- 미리보기 ---")
print("\n".join(code.splitlines()[:35]))

app.py OK — 제공 Excel 로딩 방식
--- 미리보기 ---
"""
농산물 가격 정보 시각화 대시보드 (실습 과제 답안)

같은 폴더에 제공된 KAMIS Excel 파일을 읽어 대시보드를 구성합니다.
- 딸기_가격정보.xlsx
- 배추_가격정보.xlsx
- 사과_가격정보.xlsx
- 수박_가격정보.xlsx
- 쌀_가격정보.xlsx

실행: streamlit run app.py
중지: Ctrl + C
"""

from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(
    page_title="농산물 가격 정보 대시보드",
    page_icon="🥬",
    layout="centered",
)

DATA_DIR = Path(__file__).resolve().parent

PRODUCT_FILES = {
    "딸기": "딸기_가격정보.xlsx",
    "배추": "배추_가격정보.xlsx",
    "사과": "사과_가격정보.xlsx",
    "수박": "수박_가격정보.xlsx",
    "쌀": "쌀_가격정보.xlsx",
}


In [6]:
# (선택) 노트북에서 Streamlit 서버 시작
import subprocess
import sys
from pathlib import Path

app_file = Path("app.py")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(app_file), "--server.headless", "true"],
)
print(f"Streamlit 시작 (PID={proc.pid})")
print("브라우저에서 http://localhost:8501 접속")

Streamlit 시작 (PID=10324)
브라우저에서 http://localhost:8501 접속


In [7]:
# (선택) Streamlit 프로세스 중지
try:
    proc.terminate()
    print("Streamlit 프로세스를 종료했습니다.")
except NameError:
    print("실행 중인 proc 변수가 없습니다. 터미널에서 Ctrl+C로 중지하세요.")

Streamlit 프로세스를 종료했습니다.
